In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2

# 1. LOAD DATA
crabdata = pd.read_csv('cleanCrabAgePrediction.csv') 



In [2]:
crabdata.head()

,Length,Diameter,Height,Age,Shucked_Weight_Ratio
0,1.4375,1.1750,0.4125,9,0.500575
1,0.8875,0.6500,0.2125,6,0.425197
2,1.0375,0.7750,0.2500,6,0.406417
3,1.1750,0.8875,0.2500,10,0.352261
4,0.8875,0.6625,0.2125,6,0.501027


In [3]:
crabdata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3893 entries, 0 to 3892
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Length                3893 non-null   float64
 1   Diameter              3893 non-null   float64
 2   Height                3893 non-null   float64
 3   Age                   3893 non-null   int64  
 4   Shucked_Weight_Ratio  3893 non-null   float64
dtypes: float64(4), int64(1)
memory usage: 152.2 KB


In [4]:
#remove height=0
crabdata = crabdata[crabdata['Height'] > 0]

In [5]:
print(crabdata.shape)

(3891, 5)


In [6]:
from sklearn.preprocessing import StandardScaler

X=crabdata[["Length","Height","Diameter","Shucked_Weight_Ratio"]].copy()

Y=crabdata["Age"].copy()



In [7]:
Y.head()

0     9
1     6
2     6
3    10
4     6
Name: Age, dtype: int64

In [8]:
X.head()

,Length,Height,Diameter,Shucked_Weight_Ratio
0,1.4375,0.4125,1.1750,0.500575
1,0.8875,0.2125,0.6500,0.425197
2,1.0375,0.2500,0.7750,0.406417
3,1.1750,0.2500,0.8875,0.352261
4,0.8875,0.2125,0.6625,0.501027


In [9]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


numerical_cols_to_scale = ["Length", "Diameter", "Height"]
numerical_cols_no_scale = ["Shucked_Weight_Ratio"]

# Define preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num_scale', StandardScaler(), numerical_cols_to_scale),
        ('num_no_scale', 'passthrough', numerical_cols_no_scale)
        
    ]
)

# Define the pipeline
pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

# Fit and transform the data
X_processed = pipeline.fit_transform(X)
print(X_processed)

[[ 0.41953843  0.62039253  0.60126404  0.50057537]
 [-1.41179319 -1.49529346 -1.3091317   0.42519685]
 [-0.91233911 -0.9915587  -0.9509325   0.40641711]
 ...
 [-2.28583783 -2.25089561 -1.78673064  0.38028169]
 [-0.82909676 -0.9915587  -0.83153277  0.43561644]
 [-1.74476257 -1.64641389 -1.3091317   0.36933798]]


In [10]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train_scaled, X_test_scaled, Y_train, Y_test = train_test_split(X_processed, Y,
                                                    train_size=0.8,
                                                    random_state=123)

# Make log transformation
Y_train_log=np.log1p(Y_train)
Y_test_transformed = np.log1p(Y_test)

In [20]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5]
}

rf_model_tuned = RandomForestRegressor()
grid_search_rf = GridSearchCV(rf_model_tuned, param_grid_rf, cv=5, scoring='neg_mean_squared_error')
grid_search_rf.fit(X_train_scaled, Y_train_log)

# Access the best parameters and best score
best_params_rf = grid_search_rf.best_params_
best_score_rf = grid_search_rf.best_score_

print("Best parameters for RandomForest:", best_params_rf)
print("Best score for RandomForest:", best_score_rf)

Best parameters for RandomForest: {'max_depth': 5, 'n_estimators': 200}
Best score for RandomForest: -0.03096911397746434


In [21]:
rf_model_best = RandomForestRegressor(**best_params_rf)
rf_model_best.fit(X_train_scaled, Y_train_log)

# Predictions on the training set
rf_train_predictions = rf_model_best.predict(X_train_scaled)

# Training MSE
rf_train_mse = mean_squared_error(Y_train_log, rf_train_predictions)

# Predictions on the test set
rf_test_predictions = rf_model_best.predict(X_test_scaled)

# Test MSE
rf_test_mse = mean_squared_error(Y_test_transformed, rf_test_predictions)

# RMSE for training predictions
rf_train_rmse = mean_squared_error(Y_train_log, rf_train_predictions, squared=False)

# RMSE for test predictions
rf_test_rmse = mean_squared_error(Y_test_transformed, rf_test_predictions, squared=False)

# MAE for training predictions
rf_train_mae = mean_absolute_error(Y_train_log, rf_train_predictions)

# MAE for test predictions
rf_test_mae = mean_absolute_error(Y_test_transformed, rf_test_predictions)

# R^2 for training predictions
rf_train_r2 = r2_score(Y_train_log, rf_train_predictions)

# R^2 for test predictions
rf_test_r2 = r2_score(Y_test_transformed, rf_test_predictions)

# Difference between train and test MSE
rf_mse_difference = rf_train_mse - rf_test_mse

# Calculate the difference between train and test MSE
rf_mse_difference = rf_train_mse - rf_test_mse

print("Random Forest Train MSE:", rf_train_mse)
print("Random Forest Test MSE:", rf_test_mse)
print("Random Forest Regression Train RMSE:", rf_train_rmse)
print("Random Forest Regression Test RMSE:", rf_test_rmse)
print("Random Forest Regression Train MAE:", rf_train_mae)
print("Random Forest Regression Test MAE:", rf_test_mae)
print("Random Forest Regression Train R^2:", rf_train_r2)
print("Random Forest Regression Test R^2:", rf_test_r2)
print("Random Forest Train-Test MSE Difference:", rf_mse_difference)

Random Forest Train MSE: 0.02735768136910448
Random Forest Test MSE: 0.02937879374703439
Random Forest Regression Train RMSE: 0.16540157607805459
Random Forest Regression Test RMSE: 0.17140243215028889
Random Forest Regression Train MAE: 0.1278076466718775
Random Forest Regression Test MAE: 0.1298494183511545
Random Forest Regression Train R^2: 0.6673240271171499
Random Forest Regression Test R^2: 0.6313977222867238
Random Forest Train-Test MSE Difference: -0.002021112377929908


In [22]:
# 1. Define your new data (Example values based on your dataset)
# Order: ['Length', 'Height', 'Diameter', 'Shucked_Weight_Ratio']
new_crab = pd.DataFrame([[1.4375, 0.4125, 1.1750, 0.500575]], 
                        columns=["Length", "Height", "Diameter", "Shucked_Weight_Ratio"])

# 2. Transform the new data using the pipeline already fitted in your notebook
new_crab_processed = pipeline.transform(new_crab)

# 3. Predict the log-age
log_age_prediction = rf_model_best.predict(new_crab_processed)

# 4. Convert log-age back to actual age (Inverse of log1p is expm1)
predicted_age = np.expm1(log_age_prediction)

print(f"Predicted Age: {predicted_age[0]:.2f}")

Predicted Age: 9.39


In [23]:
# 1. Generate log-age predictions for all processed observations
all_log_predictions = rf_model_best.predict(X_processed)

# 2. Transform the predictions back to the original age scale
# Using np.expm1 because the model was trained on np.log1p(Y)
predicted_ages_actual = np.expm1(all_log_predictions)

# 3. Add the predictions back to your 'crabdata' dataframe for comparison
crabdata['Predicted_Age'] = predicted_ages_actual

# 4. Display the original Age vs Predicted Age for the first 10 rows
print(crabdata[['Age', 'Predicted_Age']].head(100))

# 5. Optional: Calculate the overall Mean Absolute Error in years
from sklearn.metrics import mean_absolute_error
final_mae = mean_absolute_error(crabdata['Age'], crabdata['Predicted_Age'])
print(f"\nOverall average error: {final_mae:.2f} years")

    Age  Predicted_Age
0     9       9.392216
1     6       6.716713
2     6       8.006617
3    10      10.627574
4     6       6.579018
..  ...            ...
95    8       8.695603
96    8       7.603058
97   10       9.208010
98   10      10.997317
99    9      10.154960

[100 rows x 2 columns]

Overall average error: 1.46 years
